In [3]:
%pip install -q optuna torchmetrics pandas scikit-learn matplotlib numpy

import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.nn import CrossEntropyLoss
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

from torchmetrics.classification import MulticlassAccuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    drive.mount('/content/drive', force_remount=True)
    path = "/content/drive/MyDrive/gesture_ml/"
else:
    path = "/data"

print("Data path:", path)

Note: you may need to restart the kernel to use updated packages.
cuda
Data path: /data


In [12]:
import torch

print('torch:', torch.__version__)
print('torch CUDA build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda device 0:', torch.cuda.get_device_name(0))
else:
    print('CUDA is still unavailable in this kernel.')
    print('If torch version ends with +cpu, use Python 3.12 and reinstall from the CUDA index URL.')

torch: 2.11.0+cu128
torch CUDA build: 12.8
cuda available: True
cuda device count: 1
cuda device 0: NVIDIA GeForce RTX 3050 Laptop GPU


# Model

- has to be very small (<100 KB)

In [28]:
data = np.loadtxt("data/raw.txt", dtype=np.float32)

if data.ndim == 1:
    data = data.reshape(1, -1)

X = data[:, :-1]
y = data[:, -1].astype(np.int64)

classes, counts = np.unique(y, return_counts=True)
num_classes = len(classes)
print(num_classes)


n_features = X.shape[1]
n_gestures = num_classes


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=67
)

print("X shape:", X.shape, "y shape:", y.shape)
print("class counts:", dict(zip(classes.tolist(), counts.tolist())))

2
X shape: (101, 300) y shape: (101,)
class counts: {0: 47, 1: 54}


In [29]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


X_train = X_train.to(device)
X_test  = X_test.to(device)
y_train = y_train.to(device)
y_test  = y_test.to(device)

batch_size = 32
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size)

In [30]:
class MLP(nn.Module):
    def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 128),
                nn.ReLU(),
                nn.Dropout(0.15),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, n_gestures)
            )

    def forward(self, x):
        return self.net(x)

In [31]:
model = MLP().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

acc_metric = MulticlassAccuracy(num_classes=num_classes).to(device)

cel = CrossEntropyLoss()

for epoch in range(128):
    total_loss = 0

    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        y_pred = model(Xb)

        loss = cel(y_pred, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.size(0)

    model.eval()
    acc_metric.reset()

    with torch.no_grad():
        for Xb, yb in test_loader:
            preds = model(Xb)
            acc_metric.update(preds, yb)
        
    acc = acc_metric.compute().item()

    print(F"E:{epoch+1}, L:{loss.item()}, %:{acc}")


torch.save(model, "demo.pt")

E:1, L:0.6619263291358948, %:0.8136363625526428
E:2, L:0.5384787917137146, %:0.8590909242630005
E:3, L:0.4424019455909729, %:0.9045454263687134
E:4, L:0.39530590176582336, %:0.9045454263687134
E:5, L:0.3303579092025757, %:0.8545454740524292
E:6, L:0.16012240946292877, %:0.8545454740524292
E:7, L:0.13119764626026154, %:0.8999999761581421
E:8, L:0.1705780327320099, %:0.8999999761581421
E:9, L:0.13258229196071625, %:0.8999999761581421
E:10, L:0.16999658942222595, %:0.8999999761581421
E:11, L:0.13693836331367493, %:0.8999999761581421
E:12, L:0.10329790413379669, %:0.8999999761581421
E:13, L:0.09063459187746048, %:0.8999999761581421
E:14, L:0.07632963359355927, %:0.8999999761581421
E:15, L:0.07926800847053528, %:0.8999999761581421
E:16, L:0.1321677565574646, %:0.8999999761581421
E:17, L:0.04992406442761421, %:0.8999999761581421
E:18, L:0.071725994348526, %:0.8999999761581421
E:19, L:0.018544338643550873, %:0.8999999761581421
E:20, L:0.07348760217428207, %:0.8999999761581421
E:21, L:0.043914

In [32]:
from pathlib import Path
import copy
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

# Post-training dynamic quantization using torchao eager API (non-deprecated).
model_cpu = model.to("cpu").eval()
quantized_model = copy.deepcopy(model_cpu)
quantize_(quantized_model, Int8DynamicActivationInt8WeightConfig())

with torch.no_grad():
    fp32_preds = model_cpu(X_test.cpu()).argmax(dim=1)
    int8_preds = quantized_model(X_test.cpu()).argmax(dim=1)
    fp32_acc = (fp32_preds == y_test.cpu()).float().mean().item()
    int8_acc = (int8_preds == y_test.cpu()).float().mean().item()

out_dir = Path("output")
temp_dir = out_dir / "temp"
temp_dir.mkdir(parents=True, exist_ok=True)

fp32_path = temp_dir / "fp32.pt"
int8_path = out_dir / "int8.pt"
torch.save(model_cpu.state_dict(), fp32_path)
torch.save(quantized_model.state_dict(), int8_path)

fp32_size = fp32_path.stat().st_size
int8_size = int8_path.stat().st_size
compression = fp32_size / int8_size if int8_size else float("inf")

print("FP32 test acc:", round(fp32_acc, 4))
print("INT8 test acc:", round(int8_acc, 4))
print("FP32 state_dict path:", fp32_path)
print("INT8 state_dict path:", int8_path)
print("FP32 state_dict bytes:", fp32_size)
print("INT8 state_dict bytes:", int8_size)
print("Compression ratio:", round(compression, 2), "x")
print("INT8 under 100KB:", int8_size < 100 * 1024)

# Put original model back where training/inference expects it.
model = model.to(device)

FP32 test acc: 0.9048
INT8 test acc: 0.9048
FP32 state_dict path: output\temp\fp32.pt
INT8 state_dict path: output\int8.pt
FP32 state_dict bytes: 190385
INT8 state_dict bytes: 53020
Compression ratio: 3.59 x
INT8 under 100KB: True


c:\Users\onewi\grifting-goat\gesture_ml\.venv\Lib\site-packages\torchao\dtypes\utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\onewi\grifting-goat\gesture_ml\.venv\Lib\site-packages\torchao\dtypes\uintx\plain_layout.py:82: UserWarning: Deprecation: PlainAQTTensorImpl is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\onewi\grifting-goat\gesture_ml\.venv\Lib\site-packages\torchao\dtypes\affine_quantized_tensor.py:116: UserWarning: Deprecation: AffineQuantizedTensor is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
